In [ ]:
from pathlib import Path

EXPERIMENT_DIR = Path.cwd().resolve()
if not (EXPERIMENT_DIR / "data" / "reference").is_dir():
    candidate = EXPERIMENT_DIR.parent
    if (candidate / "data" / "reference").is_dir():
        EXPERIMENT_DIR = candidate
if not (EXPERIMENT_DIR / "data" / "reference").is_dir():
    raise RuntimeError("Start Jupyter in this experiment directory or its notebooks directory.")

REFERENCE_DATA_DIR = EXPERIMENT_DIR / "data" / "reference"
GENERATED_DATA_DIR = EXPERIMENT_DIR / "runs" / "generated-data"
REFERENCE_MODEL_DIR = EXPERIMENT_DIR / "models" / "reference"
TRAINED_MODEL_DIR = EXPERIMENT_DIR / "runs" / "trained-models"
PROJECT_DIR = EXPERIMENT_DIR.parents[3]
EXPERIMENT_ID = EXPERIMENT_DIR.relative_to(PROJECT_DIR / "experiments")
RESULT_DIR = PROJECT_DIR / "figures" / "paper"
for directory in (GENERATED_DATA_DIR, TRAINED_MODEL_DIR, RESULT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# True reproduces the paper. Set False to regenerate data and retrain.
REPRODUCE_PAPER = True

def data_path(filename):
    root = REFERENCE_DATA_DIR if REPRODUCE_PAPER else GENERATED_DATA_DIR
    return root / filename

def model_path(filename):
    root = REFERENCE_MODEL_DIR if REPRODUCE_PAPER else TRAINED_MODEL_DIR
    return root / filename

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error
from torch.utils.data import TensorDataset, DataLoader
from scipy.signal import savgol_filter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### Generate data

In [ ]:
# Define variables
k = 1            # spring constant
m = 1            # mass 
b = 0.02        # damping coefficient
noise_var = 0.0
omega_0 = np.sqrt(k / m)
beta = b / (2 * m)
w1 = np.sqrt(omega_0**2 - beta**2)  # Damped natural frequency

In [ ]:
def init_conditions(num_samples, energy_range=(0.2,1)):
    energies = np.random.uniform(energy_range[0], energy_range[1], num_samples)
    phis = np.random.uniform(0, 2 * np.pi, num_samples)
    As = []
    
    for E in energies:
        Amp = np.sqrt(2 * E / k)
        As.append(Amp)

    return np.array(As), phis

def position(A, phi, t):
    return A * np.exp(-beta * t) * np.cos(w1 * t + phi)

def momentum(A, phi, t):
    return -A * m * np.exp(-beta * t) * (beta * np.cos(w1 * t + phi) + w1 * np.sin(w1 * t + phi))

def get_derivative(x_curr, x_prev, time_step):
    return (x_curr - x_prev)/time_step

def compute_derivatives(values : np.ndarray, time_step):
    window = 3
    smoothed_values = savgol_filter(values, window_length=window, polyorder=2)
    return np.gradient(smoothed_values, time_step)

# Add noise to the data 
def add_noise(qs): 
    return qs + np.random.normal(0, noise_var, len(qs))
    
# Generate data based on initial conditions
def generate_spring_mass_data(num_trajectories, time_step, noisy=False, num_data=314, sample_all=False, sample_every=100, energy_range=(0.2,1)):
    As_0, phis_0 = init_conditions(num_trajectories, energy_range)
    
    qs, ts = [], []
    q_dots, q_ddots = [], []
    for As_0, phi_0 in zip(As_0, phis_0):
        t_vals = np.arange(0, num_data, time_step)

        # append position and momentum over all values of t_vals for each initial condition 
        if noisy: 
            qs_traj = add_noise(position(As_0, phi_0, t_vals))
        else:
            qs_traj = position(As_0, phi_0, t_vals)

        q_dot = compute_derivatives(qs_traj, time_step)
        q_ddot = compute_derivatives(q_dot, time_step)

        if sample_all: 
            qs.append(qs_traj[:])
            q_dots.append(q_dot[:])
            q_ddots.append(q_ddot[:])
            ts.append(t_vals[:])
        else:        
            # Sample every 100th data 
            qs.append(qs_traj[::sample_every])
            q_dots.append(q_dot[::sample_every])
            q_ddots.append(q_ddot[::sample_every])
            ts.append(t_vals[::sample_every])
    
    return np.asarray(qs), np.asarray(q_dots), np.asarray(q_ddots), np.asarray(ts)

#### Generate Training Data

In [ ]:
# Parameters
num_trajectories = 40                       # 40 sets
num_data = 314                              # Each with 100 time steps
time_step = 0.1  

# Generate data
qs, q_dots, q_ddots, ts = generate_spring_mass_data(num_trajectories, time_step, noisy=True, sample_all=False, sample_every=10)

# Reshape into single dataset
qs_flat = qs.reshape(-1)
q_dots_flat = q_dots.reshape(-1)
q_ddots_flat = q_ddots.reshape(-1)
ts_flat = ts.reshape(-1)

# Create DataFrame
train_df = pd.DataFrame({
    "q": qs_flat,
    "q_dot": q_dots_flat,
    "t": ts_flat,
    "q_ddot": q_ddots_flat
})

train_df.to_csv(GENERATED_DATA_DIR / "spring_mass_train_damped_no_noise.csv", index=False)

#### Generate Test Data

In [ ]:
# Generate data
qs, q_dots, q_ddots, ts = generate_spring_mass_data(num_trajectories, time_step, noisy=False, sample_all=True, energy_range=(0.2,0.2))

# Reshape into single dataset
qs_flat = qs.reshape(-1)
q_dots_flat = q_dots.reshape(-1)
q_ddots_flat = q_ddots.reshape(-1)
ts_flat = ts.reshape(-1)

# Create DataFrame
test_df_low = pd.DataFrame({
    "q": qs_flat,
    "q_dot": q_dots_flat,
    "t": ts_flat,
    "q_ddot": q_ddots_flat
})
test_df_low.to_csv(GENERATED_DATA_DIR / "spring_mass_test_damped_low_energy.csv", index=False)

# Generate data
qs, q_dots, q_ddots, ts = generate_spring_mass_data(num_trajectories, time_step, noisy=False, sample_all=True, energy_range=(1.0, 1.0))

# Reshape into single dataset
qs_flat = qs.reshape(-1)
q_dots_flat = q_dots.reshape(-1)
q_ddots_flat = q_ddots.reshape(-1)
ts_flat = ts.reshape(-1)

# Create DataFrame
test_df_high = pd.DataFrame({
    "q": qs_flat,
    "q_dot": q_dots_flat,
    "t": ts_flat,
    "q_ddot": q_ddots_flat
})
test_df_high.to_csv(GENERATED_DATA_DIR / "spring_mass_test_damped_high_energy.csv", index=False)

### Load Data

In [ ]:
train_df = pd.read_csv(data_path("spring_mass_train_damped_no_noise.csv"))
train_df = train_df.iloc[3:]
test_df_low = pd.read_csv(data_path("spring_mass_test_damped_low_energy.csv"))
test_df_low = test_df_low.iloc[3:]
test_df_high = pd.read_csv(data_path("spring_mass_test_damped_high_energy.csv"))
test_df_high = test_df_high.iloc[3:]

### Prepare training data 

In [ ]:
features = train_df.iloc[:, 0:3]
targets = train_df.iloc[:, 3:]

In [ ]:
# scaler = MinMaxScaler(feature_range=(-1,1))
# scaler = StandardScaler()
scaler = MinMaxScaler()
features_train_scaled = scaler.fit_transform(features)

In [ ]:
f_tensor = torch.FloatTensor(features_train_scaled).to(device)
t_tensor = torch.FloatTensor(targets.values).to(device)

In [ ]:
train_dataset = TensorDataset(f_tensor, t_tensor)
train_dl = DataLoader(train_dataset, batch_size=32, shuffle=True)

### Constants

In [ ]:
HIDDEN_DIM = 300
LEARNING_RATE = 1e-3
EPOCHS = 100
if REPRODUCE_PAPER:
    EPOCHS = 0
INPUT_DIM = 3
OUTPUT_DIM = 1 
OUTPUT_DIM_B = 1
HIDDEN_LAYERS_DIM = 7
MOMENTUM = 0.9

### Model Definitions

In [ ]:
# Build Neural Network
class BaselineNN(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, output_dim=OUTPUT_DIM_B, hidden_layers=HIDDEN_LAYERS_DIM):
        super(BaselineNN, self).__init__()
        
        self.inp_layer = nn.Linear(input_dim, hidden_dim)
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(hidden_layers)])
        self.out_layer = nn.Linear(hidden_dim, output_dim)
        self.activation = torch.tanh

    def forward(self, x):
        x = self.activation(self.inp_layer(x))
        for layer in self.hidden_layers:
            x = self.activation(layer(x))
        x = self.out_layer(x)
        return x
        
class LNN(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, output_dim=OUTPUT_DIM, hidden_layers=HIDDEN_LAYERS_DIM):
        super(LNN, self).__init__()

        self.inp_layer = nn.Linear(input_dim, hidden_dim)
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(hidden_layers)])
        self.out_layer = nn.Linear(hidden_dim, output_dim)
        self.activation = torch.tanh
        self.dropout = nn.Dropout(0.1)
        self.apply(self._init_weights)
        
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.xavier_uniform_(module.weight)
            torch.nn.init.zeros_(module.bias)

    def forward(self, x):
        x = self.activation(self.inp_layer(x))
        for layer in self.hidden_layers:
            x = self.activation(layer(x))
        x = self.out_layer(x)
        return x

    def grad_forward(self, x):
        # Ensure that x requires gradient.
        x = x.clone().detach().requires_grad_(True)
        # x.requires_grad_(True)
        L = self.forward(x)
        grad_outputs = torch.ones_like(L)

        grad_L = torch.autograd.grad(outputs=L, inputs=x, grad_outputs=grad_outputs, create_graph=True)[0]
        dl_dq = grad_L[:, 0]
        dl_dqdot = grad_L[:, 1]
        dl_dt = grad_L[:, 2]

        # Compute dl_dq_dqdot (d2l/dq.dqdot), dl_dqdot_dqdot (d2l/dqdot.dqdot), and dl_dt_dqdot (d2l/dt.dqdot)
        grad_outputs = torch.ones_like(dl_dqdot)
        grad_dl_dqdot = torch.autograd.grad(outputs=dl_dqdot, inputs=x, grad_outputs=grad_outputs, create_graph=True)[0]
        dl_dqdot_dq = grad_dl_dqdot[:, 0]
        dl_dqdot_dqdot = grad_dl_dqdot[:, 1]
        dl_dt_dqdot = grad_dl_dqdot[:, 2]

        q_dot = x[:, 1]
        epsilon = 1e-6 
        q_ddot = (dl_dq - dl_dqdot_dq * q_dot - dl_dt_dqdot) / (dl_dqdot_dqdot + epsilon)
        
        return q_ddot

In [ ]:
def train_baselineNN(data_dl, learning_rate=LEARNING_RATE, epochs=EPOCHS):
    best_model_state_baseline = None
    min_loss = float('inf')
    model = BaselineNN(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM).to(device)
    
    # Define the mean squared error loss
    # criterion = nn.MSELoss()
    criterion = nn.SmoothL1Loss() 
    
    # Create an Adam optimizer with weight decay for regularization
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    
    # Add learning rate scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=15, factor=0.5)
    
    model.train()
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        batch_count = 0
        
        for inputs, targets in data_dl:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            
            # Add gradient clipping to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            epoch_loss += loss.item()
            batch_count += 1
        
        avg_loss = epoch_loss / max(batch_count, 1)
        scheduler.step(avg_loss)
        
        if epoch % 10 == 0:
            print(f"Epoch: {epoch}/{epochs}, Avg Loss: {avg_loss:.6f}, LR: {optimizer.param_groups[0]['lr']:.2e}")

        if avg_loss < min_loss and batch_count > 0:
            min_loss = avg_loss
            best_model_state_baseline = copy.deepcopy(model.state_dict())
    
    print(f"Training completed. Best loss: {min_loss:.6f}")
    
    return model, best_model_state_baseline

def train_LNN(data_dl, learning_rate=LEARNING_RATE, epochs=EPOCHS):
    best_model_state = None
    min_loss = float('inf')
    model = LNN(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM).to(device)
    
    # criterion = nn.MSELoss()
    criterion = nn.SmoothL1Loss() 
    
    # Create optimizer with better regularization
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    
    # Add learning rate scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=15, factor=0.5)
    
    model.train()
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        batch_count = 0
        
        for inputs, targets in data_dl:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            
            try:
                # Forward pass through LNN with gradient computation
                q_ddot = model.grad_forward(inputs)
                predictions = q_ddot.unsqueeze(1)  # Shape: (batch_size, 1)
                
                # Compute loss
                loss = criterion(predictions, targets)
                
                # Backward pass
                loss.backward()
                
                # Gradient clipping to prevent exploding gradients
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                # Update parameters
                optimizer.step()
                
                epoch_loss += loss.item()
                batch_count += 1
                
            except RuntimeError as e:
                print(f"Warning: Numerical instability in batch at epoch {epoch}: {e}")
                continue

        avg_loss = epoch_loss / max(batch_count, 1)
        scheduler.step(avg_loss)

        if epoch % 10 == 0:
            print(f"Epoch: {epoch}/{epochs}, Avg Loss: {avg_loss:.6f}, LR: {optimizer.param_groups[0]['lr']:.2e}")

        if avg_loss < min_loss and batch_count > 0:
            min_loss = avg_loss
            best_model_state = copy.deepcopy(model.state_dict())
            
    print(f"Training completed. Best loss: {min_loss:.6f}")
    
    return model, best_model_state

### Train model

In [ ]:
_, best_model_state_baseline = train_baselineNN(train_dl)

In [ ]:
_, best_model_state = train_LNN(train_dl)

In [ ]:
if best_model_state_baseline is not None: 
    torch.save(best_model_state_baseline, TRAINED_MODEL_DIR / "best_model_baseline_undamped_noise.pth")

best_model_baseline = BaselineNN().to(device)
best_model_baseline.load_state_dict(torch.load(model_path("best_model_baseline_undamped_noise.pth"), map_location="cpu"), strict=False)

In [ ]:
if best_model_state is not None: 
    torch.save(best_model_state, TRAINED_MODEL_DIR / "best_model_noise.pth")

best_model_lnn = LNN().to(device)
best_model_lnn.load_state_dict(torch.load(model_path("best_model_noise.pth"), map_location="cpu"), strict=False)

### Prepare testing data 

In [ ]:
features_test = test_df_high.iloc[:, 0:3]
targets_test = test_df_high.iloc[:, 3:]

features_test_scaled = scaler.transform(features_test)
f_tensor_test = torch.FloatTensor(features_test_scaled)
t_tensor_test = torch.FloatTensor(targets_test.values)

test_dataset = TensorDataset(f_tensor_test, t_tensor_test)
test_dl_high = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
features_test = test_df_low.iloc[:, 0:3]
targets_test = test_df_low.iloc[:, 3:]

features_test_scaled = scaler.transform(features_test)
f_tensor_test = torch.FloatTensor(features_test_scaled).to(device)
t_tensor_test = torch.FloatTensor(targets_test.values).to(device)

test_dataset = TensorDataset(f_tensor_test, t_tensor_test)
test_dl_low = DataLoader(test_dataset, batch_size=32, shuffle=False)

### Plot results

In [ ]:
def evaluate_model_baseline(model, data_dl):
    model.eval()
    predictions = []
    true = []

    for test_inps, test_targets in data_dl: 
        test_inps, test_targets = test_inps.to(device), test_targets.to(device)
        outputs = model(test_inps)
        pred_batch = outputs.detach().cpu().numpy()
        true_batch = test_targets.detach().cpu().numpy()

        predictions.append(pred_batch)
        true.append(true_batch)

    y_true = np.concatenate(true, axis=0) 
    y_pred = np.concatenate(predictions, axis=0)  

    return y_pred, y_true

def evaluate_model(model, data_dl):
    model.eval()
    predictions = []
    true_values = []
    L_values = [] 

    for test_inps, test_targets in data_dl: 
        test_inps, test_targets = test_inps.to(device), test_targets.to(device)
        q_ddot_pred = model.grad_forward(test_inps)
        predictions.append(q_ddot_pred.detach().cpu().numpy())
        true_values.append(test_targets.detach().cpu().numpy())
        L_val = model(test_inps).detach().cpu().numpy()
        L_values.append(L_val)

    y_true = np.concatenate(true_values, axis=0) 
    y_pred = np.concatenate(predictions, axis=0)  
    L_vals = np.concatenate(L_values, axis=0)
    
    return y_pred, y_true, L_vals
    
def plot_qddot_time(qddot_true, qddot_pred, n=250, plot_true=True, plot_pred=True, save_path=None, save=False):
    plt.figure(figsize=(7, 5))
    
    if plot_true:
        plt.plot(qddot_true[:n], label="True", color="red")

    if plot_pred:
        plt.plot(qddot_pred[:n], label="Predicted", color="blue", linestyle="--")
        
    plt.xlabel('t', fontsize=14, fontweight='bold')
    plt.ylabel(r'$\ddot{q}$', fontsize=14, fontweight='bold')
    plt.legend(fontsize=10, frameon=False)

    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()

    if save_path and save:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', transparent=True)
    plt.show()

In [ ]:
qddot_pred_base_high, qddot_true_base_high = evaluate_model_baseline(best_model_baseline, test_dl_high)
qddot_pred_base_low, qddot_true_base_low = evaluate_model_baseline(best_model_baseline, test_dl_low)

qddot_pred_lnn_high, qddot_true_lnn_high, L_high = evaluate_model(best_model_lnn, test_dl_high)
qddot_pred_lnn_low, qddot_true_lnn_low, L_low = evaluate_model(best_model_lnn, test_dl_low)

In [ ]:
# baseline low energy
# plot_qddot_time(qddot_true_base_low, qddot_pred_base_low, n=750)

In [ ]:
# lnn low energy
plot_qddot_time(qddot_true_lnn_low, qddot_pred_lnn_low, n=750, save_path=RESULT_DIR / "f3a.png", save=True)

In [ ]:
# baseline high energy
# plot_qddot_time(qddot_true_base_high, qddot_pred_base_high, n=750)

In [ ]:
# lnn high energy
# plot_qddot_time(qddot_true_lnn_high, qddot_pred_lnn_high, n=750)

In [ ]:
def plot_lagrangian_high(test_df, L_pred, plot_true=True, plot_pred=True, n=250, save_path=None, save=False):
    q = test_df["q"].values
    q_dot = test_df["q_dot"].values
    t = np.arange(len(q)) * time_step
    L_true = 0.5 * np.exp((b/m) * t) * (m * q_dot**2 - k * q**2)
    L_pred = np.asarray(L_pred).reshape(-1)  # (N,)
    
    # only use the plotted region for scaling
    Lt = L_true[:n]
    Lp = L_pred[:n]

    L_mean = Lt.mean()
    L_std  = Lt.std() + 1e-12
    L_pred_scaled = (Lp - Lp.mean()) / (Lp.std() + 1e-12) * L_std + L_mean
    
    plt.figure(figsize=(7, 5))
    
    if plot_true:
        plt.plot(Lt, label="True", color="red")

    if plot_pred:
        plt.plot(L_pred_scaled[:n], label="Predicted", color="blue", linestyle="--")
        
    plt.xlabel('t', fontsize=14, fontweight='bold')
    plt.ylabel('Lagrangian', fontsize=14, fontweight='bold')
    plt.legend(fontsize=10, frameon=False)

    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()

    if save_path and save:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', transparent=True)
    plt.show()

def plot_lagrangian_low(test_df, L_pred, plot_true=True, plot_pred=True, n=250, save_path=None, save=False):
    q = test_df["q"].values
    q_dot = test_df["q_dot"].values
    t = np.arange(len(q)) * time_step
    L_true = 0.5 * np.exp((b/m) * t) * (m * q_dot**2 - k * q**2)
    L_pred = np.asarray(L_pred).reshape(-1)  # (N,)
    
    # only use the plotted region for scaling 
    Lt = L_true[:n]
    Lp = L_pred[:n]

    L_mean = Lt.mean()
    L_std  = Lt.std() + 1e-12
    L_pred_scaled = (Lp - Lp.mean()) / (Lp.std() + 1e-12) * L_std + L_mean

    plt.figure(figsize=(7, 5))
    
    if plot_true:
        plt.plot(L_true[:n], label="True", color="red")

    if plot_pred:
        plt.plot(L_pred_scaled[:n], label="Predicted", color="blue", linestyle="--")
        
    plt.xlabel('t', fontsize=14, fontweight='bold')
    plt.ylabel('Lagrangian', fontsize=14, fontweight='bold')
    plt.legend(fontsize=10, frameon=False)

    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()

    if save_path and save:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', transparent=True)
    plt.show()

In [ ]:
# plot_lagrangian_high(test_df_high, L_high, n=750)

In [ ]:
plot_lagrangian_low(test_df_low, L_low, n=750, save_path=RESULT_DIR / "f3c.png", save=True)